In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import stft

# 1. ЗАГРУЗКА ДАННЫХ И ПАРАМЕТРЫ ВАРИАНТА 9
# Предварительно создадим или считаем файл var09.csv. 
# Предположим, что в файле две колонки: 'time' (с) и 'signal' (В).
try:
    data = pd.read_csv('var09.csv')
    t_signal = data.iloc[:, 0].values
    u = data.iloc[:, 1].values
except FileNotFoundError:
    # Если файла нет, сгенерируем тестовый аналог для демонстрации работы кода
    print("Файл var09.csv не найден. Генерируем тестовый сигнал для проверки...")
    fs_test = 10000  # Частота дискретизации 10 кГц
    t_signal = np.arange(0, 2.0, 1/fs_test)
    # Основная гармоника 50 Гц (весь период) + вспышки 150 Гц и 300 Гц
    u = (np.sin(2 * np.pi * 50 * t_signal) + 
         0.8 * np.sin(2 * np.pi * 150 * t_signal) * (np.heaviside(t_signal - 0.4, 1) - np.heaviside(t_signal - 1.2, 1)) +
         0.5 * np.sin(2 * np.pi * 300 * t_signal) * (np.heaviside(t_signal - 0.8, 1) - np.heaviside(t_signal - 1.6, 1)))

# Определение частоты дискретизации из данных
h = t_signal[1] - t_signal[0]
fs = 1.0 / h
N = len(u)

# Параметры окна для Варианта 9
M = 1250      # Ширина окна
P = 1000      # Перекрытие
H = M - P     # Шаг окна (250 отсчетов)
Z = int(np.floor((N - P) / H)) # Количество окон

print(f"Частота дискретизации: {fs:.1f} Гц, Всего точек: {N}")
print(f"Количество окон времени (Z): {Z}")

# =====================================================================
# ПУНКТ 1.2: ОКОННОЕ ПРЕОБРАЗОВАНИЕ ФУРЬЕ (ВРУЧНУЮ)
# =====================================================================
p = np.arange(0, M)
W = np.ones(M)  # Прямоугольное окно по условию

X = np.zeros((Z, M), dtype=complex)

for k in range(Z):
    s = u[k*H : k*H + M] * W
    # Вычисление ДПФ для текущего окна k
    for m in range(M):
        X[k, m] = np.sum(s * np.exp(-2 * np.pi * 1j * m * p / M))

# Ограничение спектра до fmax для наглядности (например, 1000 Гц)
fmax = 1000
mmax = 1 + int(np.floor(fmax * (M - 1) / fs))

X_cropped = X[:, :mmax]

# Расчет физических осей координат
tt = np.arange(0, Z) * H * h  # Время (левый край окон)
ff = np.arange(0, mmax) * fs / M  # Линейные частоты

# Действительные амплитуды (масштабирующий коэф. lm = 1 для прямоугольного окна)
lm = 1.0
U_manual = lm * (2 / M) * np.abs(X_cropped)

# Построение спектрограммы (Ручной метод)
plt.figure(figsize=(10, 5))
plt.contourf(tt, ff, np.transpose(U_manual), cmap='viridis', levels=20)
plt.title('Оконное БПФ (Ручной метод — Вариант 9)')
plt.xlabel('Время, с')
plt.ylabel('Частота, Гц')
plt.colorbar(label='Амплитуда')
plt.grid(True, alpha=0.3, linestyle='--')
plt.show()

# =====================================================================
# ПУНКТ 1.6: ИСПОЛЬЗОВАНИЕ ВСТРОЕННОЙ ФУНКЦИИ SCIPY.SIGNAL.STFT
# =====================================================================
# Обратите внимание: scipy по умолчанию включает дополнение нулями (boundary/padded),
# для точного сравнения с ручным методом отключим их: boundary=None, padded=False
fx, tx, Sx = stft(u, fs=fs, window='boxcar', nperseg=M, noverlap=P, 
                  return_onesided=True, boundary=None, padded=False)

# Обрезка до fmax
idx_fmax = np.where(fx <= fmax)[0][-1] + 1
fx_cropped = fx[:idx_fmax]
Sx_cropped = Sx[:idx_fmax, :]
U_scipy = np.abs(Sx_cropped)

# Построение спектрограммы (SciPy)
plt.figure(figsize=(10, 5))
plt.contourf(tx, fx_cropped, U_scipy, cmap='plasma', levels=20)
plt.title('Оконное БПФ (scipy.signal.stft — Вариант 9)')
plt.xlabel('Время, с')
plt.ylabel('Частота, Гц')
plt.colorbar(label='Амплитуда')
plt.grid(True, alpha=0.3, linestyle='--')
plt.show()

# Проверка расхождения между методами
print("\n--- СРАВНЕНИЕ МЕТОДОВ ---")
print(f"Размерность ручной матрицы: {U_manual.shape}")
print(f"Размерность матрицы SciPy:  {U_scipy.transpose().shape}")


ModuleNotFoundError: No module named 'pandas'